# Model Optimization - Hyperparameter Tuning

Due to computational & time constraints, very little tuning is actually done in this file. It's more of a skeleton where additional hyperparameter tuning could/should be added.

In [1]:
import numpy as np
import pandas as pd
import category_encoders as ce
import pickle
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, VotingRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance

In [2]:
# Set pandas to not use scientific notation
pd.set_option('display.float_format', lambda x: '%.9f' % x)

## Prepare Data

In [3]:
class PatternPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.scaler = StandardScaler()
        self.float_features = ['queued_projects_count', 'yardage', 'previously_published_patterns', 'projects_per_day', 'price_usd', 'days_since_previous_pattern', 'yarn_weight']
        self.drop_initial = ['pattern_id', 'projects_count', 'favorites_count', 'author_id', 'estimated_revenue', 'total_languages', 'yarn_ids',  'days_since_publication']
        self.drop_later = ['craft', 'attributes_', 'pattern_source_type_names_', 'queued_projects_count', 'pattern_author']
        self.category_cols = ['supercategory', 'category', 'subcategory', 'babycategory']
        self.yarn_weight_map = {
            'Thread': 1,
            'Cobweb': 2,
            'Lace': 3,
            'Light Fingering': 4,
            'Fingering': 5,
            'Sport': 6,
            'DK': 7,
            'Worsted': 8,
            'Aran': 9,
            'Bulky': 10,
            'Super Bulky': 11,
            'Jumbo': 12
        }

    def fit(self, X, y=None):
        temp = X.copy()

        # Exponentialize yarn_weight before log scaling
        temp['yarn_weight'] = temp['yarn_weight'].map(self.yarn_weight_map).fillna(8.5)
        temp['yarn_weight'] = np.exp(temp['yarn_weight'])

        # Log transform float_features
        for feature in self.float_features:
            temp[feature] = np.log1p(temp[feature])

        # Fit scaler
        self.scaler.fit(temp[self.float_features])
        return self

    def transform(self, X):
        X = X.copy()
        
        # Drop initial columns
        X = X.drop(columns=self.drop_initial)
        
        # Exponentialize yarn_weight before log scaling
        X['yarn_weight_flexible'] = X['yarn_weight'].isnull()
        X['yarn_weight'] = X['yarn_weight'].map(self.yarn_weight_map).fillna(8.5)
        X['yarn_weight'] = np.exp(X['yarn_weight'])
        
        # Log transform float_features
        for feature in self.float_features:
            X[feature] = np.log1p(X[feature])
        
        # Scale features
        X[self.float_features] = self.scaler.transform(X[self.float_features])
        
        # Remove outliers
        for feature in self.float_features:
            X = X[np.abs(X[feature]) <= 4]
        
        # Rename columns
        X = X.rename(columns={'has_uk_terminology': 'uk_terminology', 'has_us_terminology': 'us_terminology'})
        
        # Create knit column
        X['knit'] = ((X['craft'] == 'Knitting')).astype(int)
        
        # Drop later columns
        X = X.drop(columns=self.drop_later)
        
        # Season processing
        X['season'] = X['season'].str.lower()
        X = pd.get_dummies(X, columns=['season'], drop_first=True)
        
        # Final category
        X['final_category'] = (
            X['supercategory'].fillna('') + '_' +
            X['category'].fillna('') + '_' +
            X['subcategory'].fillna('') + '_' +
            X['babycategory'].fillna('')
        )
        X['final_category'] = X['final_category'].str.strip('_')
        X = X.drop(columns=self.category_cols)
        
        return X

In [4]:
# Load data
patterns = pd.read_pickle('preprocessed_patterns.pkl')

In [5]:
# Create preprocessing pipeline
pipe = Pipeline([('prep', PatternPreprocessor())])

In [6]:
# Apply preprocessing
patterns_prepared = pipe.fit_transform(patterns)

In [7]:
# Sort and split into train/test/validation sets
patterns_prepared = patterns_prepared.sort_values('created_at').reset_index(drop=True)

train = patterns_prepared[patterns_prepared['created_at'].dt.date < pd.to_datetime('2025-3-1').date()]
test = patterns_prepared[(patterns_prepared['created_at'].dt.date >= pd.to_datetime('2025-3-1').date()) & (patterns_prepared['created_at'].dt.date <= pd.to_datetime('2025-8-31').date())]
val = patterns_prepared[(patterns_prepared['created_at'].dt.date >= pd.to_datetime('2025-9-1').date()) & (patterns_prepared['created_at'].dt.date <= pd.to_datetime('2026-2-28').date())]

In [8]:
# Prepare features and target
X_train = train.drop(columns=['projects_per_day', 'created_at'])
y_train = train['projects_per_day']

X_test = test.drop(columns=['projects_per_day', 'created_at'])
y_test = test['projects_per_day']

X_val = val.drop(columns=['projects_per_day', 'created_at'])
y_val = val['projects_per_day']

In [9]:
# Target encode final_category
encoder = ce.CatBoostEncoder(cols=['final_category'])
encoder.fit(X_train['final_category'], y_train)

X_train['final_category'] = encoder.transform(X_train['final_category'])
X_test['final_category'] = encoder.transform(X_test['final_category'])
X_val['final_category'] = encoder.transform(X_val['final_category'])

In [10]:
# Load top features and filter datasets
with open('top_features_for_modeling.pkl', 'rb') as f:
    top_features_df = pickle.load(f)

top_features = top_features_df[0].tolist()

X_train = X_train[top_features]
X_test = X_test[top_features]
X_val = X_val[top_features]

## Ada Boost Hyperparameter Tuning

In [11]:
# Define parameter grid for AdaBoost
param_grid_ab = {
    'n_estimators': [50],
    'learning_rate': [1.0],
    'loss': ['linear', 'square']
}

# Initialize AdaBoost Regressor
ab = AdaBoostRegressor(random_state=42026)

# Set up Grid Search with cross-validation
grid_search_ab = GridSearchCV(
    estimator=ab,
    param_grid=param_grid_ab,
    cv=3,  # Reduced from 4 to 3 folds & removed some parameter options after kernel crashed
    scoring='r2',
    n_jobs=1,
    verbose=2
)

# Fit the grid search on training data
grid_search_ab.fit(X_train, y_train)

# Print best parameters and score
print('Best parameters found for AdaBoost: ', grid_search_ab.best_params_)
print('Best cross-validation score for AdaBoost: ', grid_search_ab.best_score_)

# Evaluate on test set
best_ab = grid_search_ab.best_estimator_
test_score_ab = best_ab.score(X_test, y_test)
print('AdaBoost test set score: ', test_score_ab)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV] END ....learning_rate=1.0, loss=linear, n_estimators=50; total time= 2.1min
[CV] END ....learning_rate=1.0, loss=linear, n_estimators=50; total time= 1.1min
[CV] END ....learning_rate=1.0, loss=linear, n_estimators=50; total time= 1.9min
[CV] END ....learning_rate=1.0, loss=square, n_estimators=50; total time= 4.8min
[CV] END ....learning_rate=1.0, loss=square, n_estimators=50; total time= 2.0min
[CV] END ....learning_rate=1.0, loss=square, n_estimators=50; total time= 4.3min
Best parameters found for AdaBoost:  {'learning_rate': 1.0, 'loss': 'linear', 'n_estimators': 50}
Best cross-validation score for AdaBoost:  -0.6095130535834633
AdaBoost test set score:  0.05634998728046692


### Permutation Importances

In [12]:
result = permutation_importance(
	best_ab,
	X_test,
	y_test,
	random_state = 42026
)

ab_perm = pd.DataFrame({
	'feature': X_train.columns,
	'importance_mean': result.importances_mean,
	'importance_std': result.importances_std
}).sort_values('importance_mean', ascending = False)

In [ ]:
print(ab_perm.head(15)) # 15+ are unimportant

                                feature  importance_mean  importance_std
53                                 knit      0.084408795     0.005832129
80                            price_usd      0.076907192     0.002620977
95          days_since_previous_pattern      0.003614635     0.000996953
44                    yarn_fiber_Merino      0.002900027     0.000137559
65                  attributes_top-down      0.002782893     0.003632725
29                         animal_fiber      0.002091453     0.000152452
98            attributes_video-tutorial      0.001735505     0.000349934
77                   yarn_fiber_Acrylic      0.000586220     0.000150124
64                      synthetic_fiber      0.000520426     0.000129581
103            attributes_positive-ease      0.000319569     0.000102931
117  pattern_source_type_names_Pamphlet      0.000316619     0.000054730
50                          yarn_weight      0.000214379     0.000395641
2                  attributes_bottom-up      0.0001

In [30]:
top_features_ab = ab_perm.head(14)['feature'].tolist()

## Random Forest Hyperparameter Tuning

In [15]:
# Define parameter grid for Random Forest
param_grid = {
    'n_estimators': [50],
    'max_depth': [15],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt'] # `None` gives good results but crashes kernel
}

# Initialize Random Forest Regressor
rf = RandomForestRegressor(random_state=42026)

# Set up Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=2
)

# Fit the grid search on training data
grid_search.fit(X_train, y_train)

# Print best parameters and score
print('Best parameters found: ', grid_search.best_params_)
print('Best cross-validation score: ', grid_search.best_score_)

# Evaluate on test set
best_rf = grid_search.best_estimator_
test_score = best_rf.score(X_test, y_test)
print('Test set score: ', test_score)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=50; total time=  50.7s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=50; total time=  49.8s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=50; total time=  50.3s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=50; total time=  50.1s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=50; total time=  50.6s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=50; total time=  50.7s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=  50.2s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total t

### Permutation Importances

In [16]:
result = permutation_importance(
	best_rf,
	X_test,
	y_test,
	random_state = 42026
)

rf_perm = pd.DataFrame({
	'feature': X_train.columns,
	'importance_mean': result.importances_mean,
	'importance_std': result.importances_std
}).sort_values('importance_mean', ascending = False)

In [ ]:
print(rf_perm.head(125)) # 125+ are unimportant

                    feature  importance_mean  importance_std
80                price_usd      0.051535827     0.001185648
5            final_category      0.039795836     0.000861725
53                     knit      0.028930152     0.001200237
65      attributes_top-down      0.028541972     0.000781309
81      attributes_seamless      0.016288160     0.000614158
..                      ...              ...             ...
56         attributes_darts      0.000015267     0.000002645
34        attributes_fringe      0.000013546     0.000008153
19   attributes_square-neck      0.000012586     0.000012378
30     attributes_pineapple      0.000009360     0.000001207
125   attributes_low-vision      0.000008116     0.000021915

[125 rows x 3 columns]


In [32]:
top_features_rf = rf_perm.head(124)['feature'].tolist()

## Gradient Boosting Hyperparameter Tuning

In [19]:
# Define parameter grid for Gradient Boosting
param_grid_gb = {
    'n_estimators': [100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3],
    'subsample': [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=42026)

grid_search_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=2
)

grid_search_gb.fit(X_train, y_train)

print('Best parameters found for Gradient Boosting: ', grid_search_gb.best_params_)
print('Best cross-validation score for Gradient Boosting: ', grid_search_gb.best_score_)

best_gb = grid_search_gb.best_estimator_
test_score_gb = best_gb.score(X_test, y_test)
print('Gradient Boosting test set score: ', test_score_gb)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time= 3.7min
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time= 3.6min
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time= 3.7min
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=1.0; total time= 4.6min
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=1.0; total time= 4.5min
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=1.0; total time= 4.4min
[CV] END learning_rate=0.1, max_depth=3, n_estimators=100, subsample=0.8; total time= 3.7min
[CV] END learning_rate=0.1, max_depth=3, n_estimators=100, subsample=0.8; total time= 3.6min
[CV] END learning_rate=0.1, max_depth=3, n_estimators=100, subsample=0.8; total time= 3.7min
[CV] END learning_rate=0.1, max_depth=3, n_estimators=100, subsample=1.0; total time= 4.4min
[CV]

In [20]:
result = permutation_importance(
    best_gb,
    X_test,
    y_test,
    random_state=42026
)

gb_perm = pd.DataFrame({
    'feature': X_train.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values('importance_mean', ascending=False)

In [ ]:
print(gb_perm.head(61)) # 61+ are unimportant

                         feature  importance_mean  importance_std
80                     price_usd      0.077546448     0.001483160
65           attributes_top-down      0.026117782     0.000360624
5                 final_category      0.024535205     0.000751826
95   days_since_previous_pattern      0.012947138     0.000800820
53                          knit      0.012091759     0.000512135
..                           ...              ...             ...
127   attributes_captioned-video      0.000053047     0.000015548
25        attributes_post-stitch      0.000029670     0.000004028
6           attributes_oversized      0.000004544     0.000006232
66           attributes_selvedge      0.000003327     0.000001073
70            attributes_sleeves      0.000002580     0.000002657

[61 rows x 3 columns]


In [35]:
top_features_gb = gb_perm.head(60)['feature'].tolist()

## K-Nearest Neighbors Hyperparameter Tuning

In [23]:
param_grid_knn = {
    'n_neighbors': [5],
    'weights': ['distance'],
    'p': [1, 2]
}

knn = KNeighborsRegressor()

grid_search_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=2
)

grid_search_knn.fit(X_train, y_train)

print('Best parameters found for KNN: ', grid_search_knn.best_params_)
print('Best cross-validation score for KNN: ', grid_search_knn.best_score_)

best_knn = grid_search_knn.best_estimator_
test_score_knn = best_knn.score(X_test, y_test)
print('KNN test set score: ', test_score_knn)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV] END ..............n_neighbors=5, p=1, weights=distance; total time=107.1min
[CV] END ..............n_neighbors=5, p=1, weights=distance; total time=106.3min
[CV] END ..............n_neighbors=5, p=1, weights=distance; total time=100.3min
[CV] END ...............n_neighbors=5, p=2, weights=distance; total time=20.3min
[CV] END ...............n_neighbors=5, p=2, weights=distance; total time=20.1min
[CV] END ...............n_neighbors=5, p=2, weights=distance; total time=20.1min
Best parameters found for KNN:  {'n_neighbors': 5, 'p': 1, 'weights': 'distance'}
Best cross-validation score for KNN:  -0.012156695645727559
KNN test set score:  0.17656755385063116


## Final Feature Set

In [36]:
top_features_final = list(set(top_features_ab + top_features_rf + top_features_gb))

In [37]:
# Compare feature overlap across model importance lists
feature_sets = {
    'ab': set(top_features_ab),
    'rf': set(top_features_rf),
    'gb': set(top_features_gb)
}

feature_counts = {
    feature: sum(feature in features for features in feature_sets.values())
    for feature in set().union(*feature_sets.values())
}

features_by_count = pd.DataFrame(
    sorted(feature_counts.items(), key=lambda x: (-x[1], x[0])),
    columns=['feature', 'count']
)

print('Features in all 3 model lists:')
for feature in (features_by_count[features_by_count['count'] == 3]['feature'].tolist()):
	print(feature)

print('\nFeatures in 2 model lists:')
for feature in (features_by_count[features_by_count['count'] == 2]['feature'].tolist()):
	print(feature)

print('\nFeatures in only 1 list:')
for feature in (features_by_count[features_by_count['count'] == 1]['feature'].tolist()):
	print(feature)

Features in all 3 model lists:
animal_fiber
attributes_bottom-up
attributes_positive-ease
attributes_top-down
attributes_video-tutorial
days_since_previous_pattern
free_patterns
knit
pattern_source_type_names_Pamphlet
price_usd
synthetic_fiber
yarn_fiber_Acrylic
yarn_fiber_Merino
yarn_weight

Features in 2 model lists:
attributes_asymmetric
attributes_bias
attributes_brioche-tuck
attributes_captioned-video
attributes_child
attributes_cropped
attributes_drop-sleeve
attributes_female
attributes_icord
attributes_icord-edging
attributes_in-the-round
attributes_male
attributes_mesh
attributes_mosaic
attributes_negative-ease
attributes_other-heel
attributes_oversized
attributes_plus
attributes_post-stitch
attributes_ribbed
attributes_schematic
attributes_seamed
attributes_seamless
attributes_short-rows
attributes_slipped-stitches
attributes_straight
attributes_stranded
attributes_teen
attributes_textured
attributes_top-cuff-down
attributes_triangle-shaped
attributes_twisted-stitches
attribut

## Voting Regressor Combination

In [38]:
# Standard VotingRegressor
final_model = VotingRegressor(
	estimators = [
		('rf', best_rf),
		('adr', best_ab),
		('gb', best_gb),
		('knn', best_knn)
	]
)

final_model.fit(X_train[top_features_final], y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('rf', ...), ('adr', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_feat

In [39]:
final_model.score(X_test[top_features_final], y_test)

0.22115085649237143

In [40]:
final_model.score(X_val[top_features_final], y_val)

0.10682982246221917